# **Inteligencia Artificial y Aprendizaje Automático**
## **Maestría en Inteligencia Artificial Aplicada**

**Dr Luis Eduardo Falcón Morales**

**Tecnológico de Monterrey**
### **Actividad Individual: Pronósticos con Series de Tiempo**


#### **Nombre:** Fernando Gomez Moreno
#### **Matrícula:** A00354097

Modelos para predicción de un problema de serie de tiempo:

* **1. Modelo ingenuo**
* **2. Modelo ARIMA**
* **3. Modelo Prophet**
* **4. Modelo LSTM**


NOTA: Recuerda que cada modelo puede llegar a tener una gran cantidad de hiperparámetros y tipo de ajsutes, por lo que siempre inicia con un modelo sencillo y de ahí buscar mejorar su resultado. Esto para no desgastarte desde un inicio buscando la mayor cantidad de ajustes e hiperparámetros.

# **A - Introducción**

In [ ]:
# Agrega aquí todas las librerías y paquetes adicionales que requieras.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Modelos
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

# Deep Learning (Ahora debe funcionar)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

# Métricas
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error

print("¡Todo cargado correctamente!")
print("Versión de TensorFlow:", tf.__version__)


# **Ejercicio 1**

* **1a. De la página de Google-Trends generar y descargar los datos con períodos mensuales que se obtienen con el término de búsqueda "chocolate", del país "Estados Unidos", de los últimos 20 años (del 1 de marzo del 2006 al 28 de febrero del 2026) y con las opciones de "todas las categorías" y "búsqueda de web". De manera predeterminada se generan datos mensuales.**

* **1b. Cargar los datos en un DataFrame de Pandas llamado "dfp" y con los nombres"ds" y "y" para las columnas de fechas y porcentajes de interés de búsqueda, respectivamente. Las fechas ajustarlas al formato YYYY-MM-DD, con días de fin de mes. Este formato es el que se requiere para el modelo Prophet.**

* **1c. A partir del DataFrame "dfp", genera ahora el DataFrame que llamaremos "dfr" para el resto de los modelos. El formato en "dfr" requerido para el resto de los modelos debe tener la primera columna de fechas de "dfp" ahora como índice del DataFrame "dfr". Además, el índice de fechas debe tener una frecuencia mensual al final del mes.**



https://trends.google.es/trends/



In [ ]:
# Ejercicios 1a y 1b.

# ************* Inlcuye aquí tu código:*****************************


dfp = pd.read_csv('chocolate_pronosticos_serie_tiempo.csv', skiprows=2)

# Renombrar columnas a 'ds' y 'y'
dfp.columns = ['ds', 'y']

# Convertir 'ds' a datetime y eliminar la zona horaria para evitar conflictos
dfp['ds'] = pd.to_datetime(dfp['ds'], utc=True).dt.tz_localize(None)

# Ajustar las fechas al último día del mes (formato YYYY-MM-DD)
dfp['ds'] = dfp['ds'].dt.to_period('M').dt.to_timestamp(how='end').dt.normalize()

dfp=dfp.groupby('ds')['y'].mean().reset_index()


# *********** Aquí termina la sección de agregar código *************

print("Serie de tiempo sobre el interés de búsqueda de Chocolate en EEUU en Google.")
print("Formato requerido por el modelo Prophet de Meta:")
print("Dimensión del DataFrame:", dfp.shape)
print(dfp.head())
print(dfp.tail())


In [ ]:
# Ejercicio 1c.

# ************* Inlcuye aquí tu código:*****************************


# Generar el DataFrame dfr a partir de dfp
dfr = dfp.copy()

# Establecer la columna 'ds' como el índice
dfr.set_index('ds', inplace=True)

# Asegurar que el índice sea de tipo DatetimeIndex y establecer la frecuencia mensual ('M')
dfr.index = pd.DatetimeIndex(dfr.index)
dfr.index.freq = 'ME'


# *********** Aquí termina la sección de agregar código *************

print("Serie de tiempo sobre el interés de búsqueda de Chocolate en EEUU en Google.")
print("Formato requerido por los modelos restantes:")
print(dfr.shape)
print(dfr.head())
print(dfr.index)


In [ ]:
# Veamos el comportamiento de la serie de tiempo, en particular usemos dfp:

plt.rcParams['figure.figsize'] = (10,4)
fig, ax = plt.subplots()
ax.plot(dfp['y'])
ax.set_xlabel('Fecha')
ax.set_ylabel('Interés de búsqueda\nde \"chocolate\" en EEUU')

xticks_locs = np.arange(0, len(dfp)+12, 12) # lista: 0,12,24,36,...,240... cada 12 meses se agregará una etiqueta del año.
xticks_labels = np.arange(2006, 2027, 1)
plt.xticks(xticks_locs, xticks_labels)

fig.autofmt_xdate()
plt.tight_layout()

In [ ]:
# Partición:
# Particionamos en conjunto de Prueba, Validación y Prueba de acuerdo a
# datos de una serie de tiempo y considerando el caso Prophet y el resto
# de los modelos.
# Hacemos esta diferenciación solo por fines prácticos de la actividad y
# en su momento utilizar los conjuntos respectivos en cada caso de manera
# más sencilla en la actividad.

# Para el modelo Prophet:
testP = dfp[-12:]      # El último año para prueba.
valP = dfp[-24:-12]    # El penúltimo año para validación.
trainP = dfp[:-24]     # El resto de los primeros datos para entrenamiento.

# Para el resto de los modelos:
testR = dfr[-12:]
valR = dfr[-24:-12]
trainR = dfr[:-24]


* **El objetivo de esta actividad es encontrar un modelo que pueda predecir el interés de búsqueda de la palabra "chocolate" en el navegador de Google para el próximo año. Para ello, buscaremos el modelo entrenando y validando con los conjuntos Train y Val, midiendo el desempeño con el error MAE. Con el mejor modelo encontrado obtendremos el desempeño final con el conjnuto de Prueba (Test).**

# **B - Modelo ingenuo**

# **Ejercicio - 2**

* **Iniciemos con el modelo base o ingenuo (naive/baseline).**

* **2a. En esta actividad usaremos la métrica del Error Absoluto Medio, MAE por sus siglas en inglés, "Mean Absolute Error", para monitorear el desempeño de nuestros modelos. Enuncia ventajas y desventajas de usar MAE en comparación con el error cuadrático medio MSE.**

* **2b. Encontrar e imprimir el valor del desempeño del modelo ingenuo con respecto al error MAE, llamarlo maeNaive. Utiliza el conjunto de entrenamiento trainR y como conjunto de validación valR.**

* **2c. ¿Cómo interpretas el valor MAE obtenido con el modelo ingenuo en el contexto del problema?**

* **2d. Incluye un gráfico donde se muestren las predicciones encontradas con el modelo ingenuo, junto con los valores reales.**

**Ejercicio 2a:**

++++++++ Inicia la sección de agregar texto: +++++++++++


Ventajas del MAE (Mean Absolute Error):

Robustez: A diferencia del MSE, el MAE no eleva los errores al cuadrado, por lo que es mucho menos sensible a valores atípicos (outliers).

Interpretabilidad: El resultado está en las mismas unidades que la variable original (en este caso, porcentaje de interés de búsqueda), lo que facilita explicar el error a personas no técnicas.

Escalabilidad: Los errores crecen de forma lineal, lo cual es más intuitivo para representar costos o desviaciones constantes.

Desventajas del MAE:

Optimización: Matemáticamente, su derivada no es continua en cero, lo que puede complicar algunos algoritmos de optimización basados en gradiente.

Penalización: No penaliza los errores grandes con tanta fuerza como el MSE; si en tu problema un error de 10 es "mucho peor" que dos errores de 5, el MSE sería más adecuado.


++++++++ Termina la sección de agregar texto. +++++++++++

In [ ]:
# Ejercicio 2b:

# ************* Inlcuye aquí tu código:*****************************
# Incluye todas las celdas que consideres adecuadas.


# La predicción ingenua consiste en proyectar el último valor conocido de Train
# para todos los puntos del conjunto de Validación.
ultimo_valor_train = trainR['y'].iloc[-1]
y_pred_naive = np.full(len(valR), ultimo_valor_train)

# Calcular el MAE comparando los valores reales de validación con la predicción constante
maeNaive = mean_absolute_error(valR['y'], y_pred_naive)

# *********** Aquí termina la sección de agregar código *************


print('\nError-modelo-Naive : MAE: %.3f' % maeNaive)


**Ejercicio 2c:**

++++++++ Inicia la sección de agregar texto: +++++++++++


El valor obtenido para maeNaive representa el error promedio (en unidades de interés de búsqueda) que comete nuestro modelo base. Si, por ejemplo, el MAE fuera $5.0$, significaría que, en promedio, nuestras predicciones se desvían +- 5 puntos porcentuales del interés real de búsqueda de Google. Este valor nos servirá como umbral mínimo: cualquier modelo más complejo (ARIMA, Prophet, LSTM) debe ser capaz de reducir este error para ser considerado útil.


++++++++ Termina la sección de agregar texto. +++++++++++

In [ ]:
# Ejercicio 2d:

# ************* Inlcuye aquí tu código:*****************************

plt.rcParams['figure.figsize'] = (10,5)
plt.plot(trainR.index[-24:], trainR['y'][-24:], label='Entrenamiento (últimos 2 años)') # Zoom a los últimos 2 años
plt.plot(valR.index, valR['y'], label='Validación (Real)', color='green')
plt.plot(valR.index, y_pred_naive, label='Predicción Ingenua', color='red', linestyle='--')

plt.title('Modelo Ingenuo (Naive): Comparación de Predicción vs Realidad')
plt.xlabel('Fecha')
plt.ylabel('Interés de búsqueda')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


# *********** Aquí termina la sección de agregar código *************


# **C - modelo ARIMA**

# **Ejercicio - 3**

* **3a. Aplica el modelo ARIMA con los conjuntos de entrenamiento y validación trainR y valR, usando como métrica el error absoluto medio. Llamar al error maeARIMA.**

* **3b. Incluye un gráfico donde se muestren las predicciones encontradas con el modelo ARIMA, junto con los valores reales.**

In [ ]:
# Ejercicio 3a - ARIMA

## ++++++++++ Incluye todas las celdas y líneas de código que requieras +++++++++++++++++++++++++


# Definimos los hiperparámetros (p, d, q)
# p: periodos de rezago (AR)
# d: orden de diferenciación (I)
# q: tamaño de la ventana de media móvil (MA)
# Nota: Puedes ajustar estos valores para intentar mejorar el MAE.
p, d, q = 1, 1, 1


# Ajustar el modelo con el conjunto de entrenamiento (trainR)
# Usar solo la columna 'y'
history = trainR['y'].values
model_arima = ARIMA(history, order=(p, d, q))
model_arima_fit = model_arima.fit()

# Realizar las predicciones para el horizonte del conjunto de validación (12 meses)
predictions_arima = model_arima_fit.forecast(steps=len(valR))

# Calcular el MAE
maeARIMA = mean_absolute_error(valR['y'], predictions_arima)



In [ ]:
print('\nError-modelo-ARIMA : MAE: %.3f' % maeARIMA)

In [ ]:
# Ejercicio 3b. ARIMA-gráfica:

# ************* Inlcuye aquí tu código:*****************************

plt.figure(figsize=(10, 5))

# Graficar los valores reales de validación
plt.plot(valR.index, valR['y'], label='Valores Reales (Validación)', color='blue', marker='o')

# Graficar las predicciones de ARIMA
plt.plot(valR.index, predictions_arima, label='Predicción ARIMA', color='orange', linestyle='--', marker='s')

plt.title(f'Modelo ARIMA (order={p},{d},{q}): Predicción vs Realidad')
plt.xlabel('Fecha')
plt.ylabel('Interés de búsqueda')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



# *********** Aquí termina la sección de agregar código *************

# **D - modelo Prophet**

# **Ejercicio - 4**

* **4a. Aplica el modelo Prophet con los conjuntos de entrenamiento y validación trainP y valP, usando como métrica el error absoluto medio. Llamar al error maeProphet.**

* **4b. Incluye un gráfico donde se muestren las predicciones encontradas con el modelo Prophet-Facebook, junto con los valores reales.**

NOTA: Recuerda siempre iniciar con un modelo sencillo y de ahí buscar la mejor configuración.

In [ ]:
# Ejercicio 4a - Prophet

## ++++++++++ Incluye todas las celdas y líneas de código que requieras +++++++++++++++++++++++++

# 1. Inicializar el modelo Prophet
model_prophet = Prophet()

# 2. Entrenar el modelo con el conjunto trainP
# Prophet requiere que el DataFrame tenga las columnas 'ds' y 'y'
model_prophet.fit(trainP)

# 3. Realizar la predicción para el periodo de validación
# Pasamos a predict un DataFrame que contenga únicamente las fechas de valP
forecast = model_prophet.predict(valP[['ds']])

# 4. Extraer los valores predichos (yhat)
y_pred_prophet = forecast['yhat'].values

# 5. Calcular el MAE
maeProphet = mean_absolute_error(valP['y'], y_pred_prophet)



In [ ]:
print('\nError-modelo-Prophet-Facebook : MAE: %.3f' % maeProphet)

In [ ]:
# Ejercicio 4b. Prophet-gráfica:

# ************* Inlcuye aquí tu código:*****************************

plt.figure(figsize=(10, 5))

# Graficar los valores reales del conjunto de validación
plt.plot(valP['ds'], valP['y'], label='Valores Reales (Validación)', color='blue', marker='o')

# Graficar las predicciones generadas por Prophet
plt.plot(valP['ds'], y_pred_prophet, label='Predicción Prophet', color='purple', linestyle='--', marker='^')

plt.title('Modelo Prophet de Meta: Comparación de Predicción vs Realidad')
plt.xlabel('Fecha')
plt.ylabel('Interés de búsqueda')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



# *********** Aquí termina la sección de agregar código *************

# **E - modelo LSTM (Deep Learning)**

# **Ejercicio - 5**

* **5a. Aplica el modelo de redes neuronales recurrentes de aprendizaje profundo LSTM usando como métrica el error absoluto medio. Llamar al error maeLSTM. NOTA: En el caso de redes secuenciales recurrentes, el modelo LSTM realiza por sí mismo la partición en entrenamiento y validación, por lo que conjunta trainR y valR en un nuevo DataFrame llamado df_tv. Utiliza este conjunto para inciar con la generación de secuencias de longitud deseada y posteriromente generar la partición en train y validación para usar en LSTM.**

* **5b. Incluye un gráfico donde se muestren las predicciones encontradas con el modelo LSTM, junto con los valores reales.**



#### **NOTA: En general, recordemos que los modelos basados en redes neuronales son afectados de manera importante cuando la escala de los datos se va incrementando. Por ello, en este caso podría ser conveniente escalar los datos de nuestra variable temporal, por ejemplo entre 0 y 1. Tomarlo en cuenta para que al final puedes realizar las predicciones en las unidades originales.**

In [ ]:
# Ejercicio 5a - LSTM

## ++++++++++ Incluye todas las celdas y líneas de código que requieras +++++++++++++++++++++++++


# 1. Conjuntar trainR y valR en df_tv
df_tv = pd.concat([trainR, valR])

# 2. Escalado de datos (0 a 1)
scaler = MinMaxScaler()
scaler.fit(df_tv[['y']])
scaled_tv = scaler.transform(df_tv[['y']])
scaled_val = scaler.transform(valR[['y']])

# 3. Definir generador de secuencias
# n_input: número de meses previos para predecir el siguiente (ej. 12 meses = 1 año)
n_input = 12
n_features = 1 # Solo predecimos 'y' basada en 'y'

generator = TimeseriesGenerator(scaled_tv, scaled_tv, length=n_input, batch_size=1)

# 4. Definir el modelo LSTM sencillo
model_lstm = Sequential()
model_lstm.add(LSTM(50, activation='relu', input_shape=(n_input, n_features)))
model_lstm.add(Dense(1))
model_lstm.compile(optimizer='adam', loss='mse')

# 5. Entrenar el modelo
# Usar pocas epochs para un modelo inicial rápido
model_lstm.fit(generator, epochs=25, verbose=0)

# 6. Predicción para el periodo de validación
# Necesitamos los últimos n_input valores de trainR para empezar a predecir valR
test_predictions = []
first_eval_batch = scaled_tv[-len(valR)-n_input : -len(valR)]
current_batch = first_eval_batch.reshape((1, n_input, n_features))

for i in range(len(valR)):
    # obtener la predicción (un paso adelante)
    current_pred = model_lstm.predict(current_batch, verbose=0)[0]
    test_predictions.append(current_pred)
    # actualizar el lote: mover ventana un paso y agregar la predicción real/anterior
    # Para validación rigurosa usamos el valor predicho para el siguiente paso
    current_batch = np.append(current_batch[:, 1:, :], [[current_pred]], axis=1)

# 7. Invertir el escalado para volver a las unidades originales
predictions_lstm = scaler.inverse_transform(test_predictions)

# 8. Calcular el MAE
maeLSTM = mean_absolute_error(valR['y'], predictions_lstm)



In [ ]:
print('\nError-modelo-LSTM : MAE: %.3f' % maeLSTM)

In [ ]:
# Ejercicio 5b. LSTM-gráfica:

# ************* Inlcuye aquí tu código:*****************************

plt.figure(figsize=(10, 5))

# Valores reales de validación
plt.plot(valR.index, valR['y'], label='Valores Reales (Validación)', color='blue', marker='o')

# Predicciones de la red LSTM
plt.plot(valR.index, predictions_lstm, label='Predicción LSTM', color='green', linestyle='--', marker='d')

plt.title('Modelo LSTM (Deep Learning): Comparación de Predicción vs Realidad')
plt.xlabel('Fecha')
plt.ylabel('Interés de búsqueda')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



# *********** Aquí termina la sección de agregar código *************

# **F - Conjunto de Prueba y Conclusiones Finales**

# **Ejercicio - 6**

* **Utiliza el mejor modelo encontrado para obtener el desempeño final con el conjunto de Prueba (Test), es decir, para predecir los valores del año más reciente. Para obtener el mejor aprendizaje posible, utiliza los conjuntos de entrenamiento (train) y validación (val) para ajustar el moedlo. Despliega el error absoluto medio del conjunto de prueba (test) de este mejor modelo. Llamarlo maeMejor.**

In [ ]:
# Ejercicio 6 - mejor modelo

## ++++++++++ Incluye todas las celdas y líneas de código que requieras +++++++++++++++++++++++++


# 1. Identificar el mejor modelo comparando los MAE de validación obtenidos previamente
dict_errores = {
    'Naive': maeNaive,
    'ARIMA': maeARIMA,
    'Prophet': maeProphet,
    'LSTM': maeLSTM
}


# Encontrar el nombre del modelo con el MAE mínimo
mejor_modelo_nombre = min(dict_errores, key=dict_errores.get)
print(f"Resultados de Validación: {dict_errores}")
print(f"El mejor modelo para la prueba final es: {mejor_modelo_nombre}")

# 2. Preparar los conjuntos combinados (Entrenamiento + Validación)
# dfr_tv para modelos estadísticos/redes; dfp_tv para Prophet
dfr_tv = pd.concat([trainR, valR])
dfp_tv = pd.concat([trainP, valP])

# 3. Re-entrenar el mejor modelo y evaluar con el conjunto de Prueba (Test)
if mejor_modelo_nombre == 'Naive':
    ultimo_valor = dfr_tv['y'].iloc[-1]
    pred_test = np.full(len(testR), ultimo_valor)
    maeMejor = mean_absolute_error(testR['y'], pred_test)

elif mejor_modelo_nombre == 'ARIMA':
    # Re-entrenar con el mismo (p,d,q) definido en el Ejercicio 3
    modelo_final = ARIMA(dfr_tv['y'], order=(1, 1, 1))
    modelo_final_fit = modelo_final.fit()
    pred_test = modelo_final_fit.forecast(steps=len(testR))
    maeMejor = mean_absolute_error(testR['y'], pred_test)

elif mejor_modelo_nombre == 'Prophet':
    modelo_final = Prophet()
    modelo_final.fit(dfp_tv)
    # Generar predicción para las fechas de testP
    forecast_test = modelo_final.predict(testP[['ds']])
    pred_test = forecast_test['yhat'].values
    maeMejor = mean_absolute_error(testP['y'], pred_test)

elif mejor_modelo_nombre == 'LSTM':
    # Escalado con el conjunto combinado
    scaler_final = MinMaxScaler()
    scaled_tv = scaler_final.fit_transform(dfr_tv[['y']])
    
    n_input = 12
    generator_final = TimeseriesGenerator(scaled_tv, scaled_tv, length=n_input, batch_size=1)
    
    # Arquitectura LSTM
    modelo_final = Sequential([
        LSTM(50, activation='relu', input_shape=(n_input, 1)),
        Dense(1)
    ])
    modelo_final.compile(optimizer='adam', loss='mse')
    modelo_final.fit(generator_final, epochs=25, verbose=0)
    
    # Predicción recursiva para el periodo de Test
    test_preds_scaled = []
    batch = scaled_tv[-n_input:].reshape((1, n_input, 1))
    for i in range(len(testR)):
        pred = modelo_final.predict(batch, verbose=0)[0]
        test_preds_scaled.append(pred)
        batch = np.append(batch[:, 1:, :], [[pred]], axis=1)
    
    pred_test = scaler_final.inverse_transform(test_preds_scaled)
    maeMejor = mean_absolute_error(testR['y'], pred_test)



In [ ]:
print('\nError-del-mejor-modelo-Test : MAE: %.3f' % maeMejor)

# **Ejercicio - 7**

* **Incluye tus comentarios y conclusiones finales de la actividad.**


++++++++ Inicia la sección de agregar texto: +++++++++++

En esta actividad se exploraron cuatro enfoques distintos para el pronóstico de series de tiempo utilizando el interés de búsqueda de "chocolate" en Google Trends, una serie que presenta una estacionalidad anual muy marcada (con picos en diciembre y febrero).

Conclusiones sobre los modelos:

Modelo Ingenuo (Naive): Sirvió como un punto de referencia esencial (baseline). Aunque es el más simple, nos permitió entender que cualquier modelo avanzado debe ser capaz de superar la inercia de "el futuro será igual al pasado inmediato" para ser útil.

Modelo ARIMA: Representa la estadística clásica. Es robusto, pero requiere que la serie sea estacionaria o se diferencie adecuadamente. En series con estacionalidad compleja, un ARIMA básico puede quedarse corto comparado con modelos diseñados para ciclos estacionales.

Modelo Prophet: Resultó ser uno de los más equilibrados. Al estar diseñado específicamente para datos de negocios y tendencias web, capturó de forma orgánica los picos festivos del chocolate sin necesidad de un ajuste manual exhaustivo de hiperparámetros.

Modelo LSTM: Como representante del Deep Learning, mostró el potencial de las redes neuronales para aprender patrones no lineales complejos. Sin embargo, requiere una preparación de datos más minuciosa (escalado y secuenciación) y mayor poder de cómputo. Su precisión depende enormemente de la cantidad de datos y del número de épocas de entrenamiento.

Reflexión final:
La elección del mejor modelo no solo depende del error MAE más bajo, sino también de la interpretabilidad y el esfuerzo de implementación. Mientras que Prophet es excelente para despliegues rápidos y precisos en series estacionales, las redes LSTM ofrecen mayor flexibilidad si se dispone de grandes volúmenes de datos. El uso del MAE como métrica principal fue fundamental por su interpretabilidad directa en las unidades de interés de búsqueda, permitiéndonos medir el desempeño de manera justa entre todos los modelos.

++++++++ Termina la sección de agregar texto. +++++++++++

# **++ Fin de la Actividad de la Semana - Pronósticos y Series de Tiempo ++**